# Bitcoin Next-Day Candle Direction Classifier

**Goal:** predict whether tomorrow's BTC-USD daily candle will close green (`Close > Open`, label `1`) or red (label `0`), using a binary classifier trained on price-based technical features plus cross-market signals (gold, oil, USD index, equities, volatility, Ethereum, crypto-specific sentiment/funding data).

**Models compared:** XGBoost, CatBoost, LightGBM.

---

### ⚠️ Honest disclaimer
Daily crypto price direction is very hard to predict from public market data alone — financial markets are close to efficient, and even professional quant funds rarely exceed 55-60% accuracy consistently. This notebook is built as a **rigorous, portfolio-quality demonstration of a correct ML pipeline** (careful feature engineering, leakage prevention, time-aware validation, honest baselines) — **not a live trading system or investment advice**.

---

## Table of contents
1. Setup & imports
2. Data collection (yfinance)
3. Feature engineering (technical + SMC liquidity proxies + crypto-specific sentiment)
4. Target construction (leakage-safe)
5. Train / test split (time-aware)
6. Baseline
7. Model training (XGBoost, CatBoost, LightGBM)
8. Evaluation & comparison
9. Feature importance
10. Simple backtest simulation
11. Bonus: predicting next-day volatility regime (a more tractable target)
12. Conclusions & limitations


## 1. Setup & imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

import yfinance as yf
import ta  # technical analysis indicators (RSI, MACD, Bollinger Bands, ...)

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
sns.set_style("whitegrid")


## 2. Data collection

We pull daily OHLCV data for BTC-USD plus a set of cross-market assets that are commonly believed to correlate with Bitcoin's price action:

| Ticker | Asset | Why it might matter |
|---|---|---|
| `BTC-USD` | Bitcoin | the target asset itself |
| `ETH-USD` | Ethereum | largest altcoin, high correlation with BTC |
| `GC=F` | Gold futures | traditional "safe haven" asset |
| `CL=F` | Crude oil futures | global risk / inflation proxy |
| `DX-Y.NYB` | US Dollar Index (DXY) | often inversely correlated with BTC |
| `^GSPC` | S&P 500 | BTC increasingly trades like a risk asset alongside equities |
| `^IXIC` | Nasdaq Composite | tech-heavy equities, even higher correlation with BTC historically |
| `^VIX` | CBOE Volatility Index | market "fear gauge" |
| `^TNX` | US 10-Year Treasury Yield | interest rates affect risk-asset appetite |

All data comes from Yahoo Finance via `yfinance` — free, no API key required, so this notebook is fully reproducible by anyone who clones the repo. Later in this section we also pull two **crypto-specific** signals (Funding Rate and the Fear & Greed Index) from free public APIs.

In [ ]:
TICKERS = {
    "BTC-USD": "btc",
    "ETH-USD": "eth",
    "GC=F": "gold",
    "CL=F": "oil",
    "DX-Y.NYB": "dxy",
    "^GSPC": "sp500",
    "^IXIC": "nasdaq",
    "^VIX": "vix",
    "^TNX": "tnx",
}

START_DATE = "2018-01-01"
END_DATE = None  # None = up to today

raw_data = {}
for ticker, name in TICKERS.items():
    print(f"Downloading {ticker} ({name})...")
    df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    if df.empty:
        print(f"  ⚠ WARNING: no data returned for {ticker}, skipping.")
        continue
    if isinstance(df.columns, pd.MultiIndex):
        # Newer yfinance versions return MultiIndex columns (field, ticker) even for a
        # single ticker -- flatten to just the field name (Open/High/Low/Close/Volume).
        df.columns = df.columns.get_level_values(0)
    df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    df.columns = [f"{name}_{c.lower()}" for c in df.columns]
    raw_data[name] = df

print("\nDownloaded assets:", list(raw_data.keys()))


In [ ]:
# Save raw data locally so the notebook can be re-run offline later
import os
os.makedirs("../data", exist_ok=True)
for name, df in raw_data.items():
    df.to_csv(f"../data/{name}_raw.csv")
print("Saved raw CSVs to ../data/")


In [ ]:
# Merge everything into a single DataFrame, aligned on date (BTC trades every day; other
# markets are closed on weekends/holidays -> those rows will have NaN for that asset).
merged = raw_data["btc"].copy()
for name, df in raw_data.items():
    if name == "btc":
        continue
    merged = merged.join(df, how="left")

# Forward-fill non-BTC assets for days they're closed (weekends/holidays) -- this uses only
# PAST values (the last known close), never future ones, so it does not leak information.
non_btc_cols = [c for c in merged.columns if not c.startswith("btc_")]
merged[non_btc_cols] = merged[non_btc_cols].ffill()

merged = merged.dropna()  # drop the initial rows before every asset has at least one value
print(merged.shape)
merged.tail()


### Crypto-specific signals: Funding Rate & Fear and Greed Index

Traditional cross-market assets (gold, oil, S&P 500, ...) only capture an indirect, macro-level
relationship with Bitcoin. These two signals come directly from crypto market behavior itself
and are commonly believed to carry more specific information:

- **Funding Rate** (Binance `BTCUSDT` perpetual futures, public API, no key needed) — paid
  every 8 hours between long and short traders. A strongly positive rate means leveraged
  traders are heavily long (sometimes a contrarian/reversal signal); strongly negative means
  heavily short.
- **Fear & Greed Index** (`alternative.me`, free public API, no key needed) — a daily 0-100
  score aggregating volatility, momentum, social media, and survey data into a single
  "market psychology" number.

Both are fetched here and, like the other cross-market features, will be **lagged by one day**
later during feature engineering to stay leakage-safe.

In [ ]:
# --- Fear & Greed Index (alternative.me) ---
try:
    fng_response = requests.get(
        "https://api.alternative.me/fng/", params={"limit": 0, "format": "json"}, timeout=30
    )
    fng_response.raise_for_status()
    fng_json = fng_response.json()["data"]
    fng_df = pd.DataFrame(fng_json)
    fng_df["date"] = pd.to_datetime(fng_df["timestamp"].astype(int), unit="s").dt.normalize()
    fng_df["fear_greed_value"] = fng_df["value"].astype(int)
    fng_df = fng_df.set_index("date")[["fear_greed_value"]].sort_index()
    print("Fear & Greed Index:", fng_df.shape, "-", fng_df.index.min().date(), "to", fng_df.index.max().date())
except Exception as e:
    print(f"⚠ WARNING: could not fetch Fear & Greed Index ({e}). Continuing without it.")
    fng_df = pd.DataFrame(columns=["fear_greed_value"])
    fng_df.index.name = "date"


In [ ]:
# --- Funding Rate (Binance BTCUSDT perpetual futures) ---
# The API returns at most 1000 records per call (funding happens every 8h, so ~333 days per
# call) -- we page backwards through history using `startTime` until no more data comes back.
def fetch_binance_funding_history(symbol="BTCUSDT"):
    all_records = []
    start_time = int(pd.Timestamp("2019-09-01").timestamp() * 1000)  # Binance perpetuals started ~Sept 2019
    end_time = int(pd.Timestamp.utcnow().timestamp() * 1000)

    while start_time < end_time:
        resp = requests.get(
            "https://fapi.binance.com/fapi/v1/fundingRate",
            params={"symbol": symbol, "startTime": start_time, "limit": 1000},
            timeout=30,
        )
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        all_records.extend(batch)
        start_time = batch[-1]["fundingTime"] + 1  # move past the last record we got
        if len(batch) < 1000:
            break  # last page

    return all_records


try:
    funding_records = fetch_binance_funding_history()
    funding_df = pd.DataFrame(funding_records)
    funding_df["date"] = pd.to_datetime(funding_df["fundingTime"], unit="ms").dt.normalize()
    funding_df["fundingRate"] = funding_df["fundingRate"].astype(float)
    # Several funding events per day (every 8h) -> average them into one daily value
    funding_df = funding_df.groupby("date")["fundingRate"].mean().to_frame("funding_rate_avg")
    print("Funding rate:", funding_df.shape, "-", funding_df.index.min().date(), "to", funding_df.index.max().date())
except Exception as e:
    print(f"⚠ WARNING: could not fetch funding rate history ({e}). Continuing without it.")
    funding_df = pd.DataFrame(columns=["funding_rate_avg"])
    funding_df.index.name = "date"


In [ ]:
# Merge both into `merged`, same left-join + forward-fill logic as the other assets
# (these APIs may have gaps on some days; ffill carries the last known value forward only).
merged = merged.join(fng_df, how="left")
merged = merged.join(funding_df, how="left")
merged[["fear_greed_value", "funding_rate_avg"]] = merged[["fear_greed_value", "funding_rate_avg"]].ffill()

print(merged.shape)
merged[["fear_greed_value", "funding_rate_avg"]].tail()


## 3. Feature engineering

### Price-based features (from BTC itself)
- Daily return, rolling volatility (7/14/30 days)
- Moving averages (SMA/EMA, 7/14/30) and their ratio to the current close (trend strength)
- RSI, MACD, Bollinger Band width & position
- Volume change
- Day of week

### Cross-asset features (gold, oil, DXY, S&P 500, Nasdaq, VIX, 10Y yield, ETH)
For every cross-asset series we use its **daily return, lagged by one day**. That means the
feature available for predicting day *t+1* only reflects information already known at the
**start** of day *t* — a deliberately conservative choice to avoid any ambiguity about
which market closed before which (BTC trades 24/7, traditional markets don't).

In [ ]:
def add_technical_features(df, price_prefix="btc"):
    close = df[f"{price_prefix}_close"]
    high = df[f"{price_prefix}_high"]
    low = df[f"{price_prefix}_low"]
    volume = df[f"{price_prefix}_volume"]

    feats = pd.DataFrame(index=df.index)

    # Returns & volatility
    feats["btc_return_1d"] = close.pct_change()
    for window in [7, 14, 30]:
        feats[f"btc_volatility_{window}d"] = feats["btc_return_1d"].rolling(window).std()

    # Moving averages (as ratio to close -> trend strength, scale-independent)
    for window in [7, 14, 30]:
        sma = close.rolling(window).mean()
        ema = close.ewm(span=window, adjust=False).mean()
        feats[f"btc_sma_{window}_ratio"] = close / sma
        feats[f"btc_ema_{window}_ratio"] = close / ema

    # RSI, MACD, Bollinger Bands (via the `ta` library)
    feats["btc_rsi_14"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    macd = ta.trend.MACD(close)
    feats["btc_macd"] = macd.macd()
    feats["btc_macd_signal"] = macd.macd_signal()
    feats["btc_macd_diff"] = macd.macd_diff()

    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    feats["btc_bb_width"] = (bb.bollinger_hband() - bb.bollinger_lband()) / close
    feats["btc_bb_position"] = (close - bb.bollinger_lband()) / (
        bb.bollinger_hband() - bb.bollinger_lband()
    )

    # Volume & range
    feats["btc_volume_change"] = volume.pct_change()
    feats["btc_high_low_range"] = (high - low) / close

    # Calendar
    feats["day_of_week"] = df.index.dayofweek

    return feats


btc_features = add_technical_features(merged, "btc")
print(btc_features.shape)
btc_features.tail()


### Smart Money Concepts (SMC) — liquidity proxies

⚠️ **Honesty note:** unlike RSI/MACD/Bollinger Bands (standard, precisely-defined technical
indicators), "liquidity", "stop hunts" and "order blocks" from the Smart Money Concepts /
ICT school of trading are **informal, interpretive ideas** without one agreed mathematical
definition. Whether they represent a real, exploitable market phenomenon or just a
retroactive story fitted onto price charts after the fact is genuinely debated. We include
them here as **experimental proxies**, clearly separated from the standard indicators above,
so results can be interpreted honestly.

The proxies we compute:
- **`dist_to_recent_high` / `dist_to_recent_low`** — how far (in %) the current close is from
  the highest high / lowest low of the past N days (a rough proxy for "closeness to a
  liquidity pool").
- **`liquidity_sweep_high` / `liquidity_sweep_low`** — a boolean flag for the classic "stop
  hunt" pattern: today's high (low) pierced above (below) the prior N-day high (low), but the
  candle closed back below (above) it — i.e. the breakout failed to hold.
- **`equal_highs_count`** — how many of the past N days' highs sit within a small tolerance of
  the window's maximum, a rough proxy for "liquidity resting at a shared price level".

In [ ]:
def add_smc_liquidity_features(df, price_prefix="btc", window=20, eq_tolerance=0.003):
    high = df[f"{price_prefix}_high"]
    low = df[f"{price_prefix}_low"]
    close = df[f"{price_prefix}_close"]

    feats = pd.DataFrame(index=df.index)

    # Prior N-day high/low, EXCLUDING today (shift(1) first) so "today" can be compared
    # against a level that was already established before today's candle formed.
    prior_high = high.shift(1).rolling(window).max()
    prior_low = low.shift(1).rolling(window).min()

    feats["btc_dist_to_recent_high"] = (prior_high - close) / close
    feats["btc_dist_to_recent_low"] = (close - prior_low) / close

    # Liquidity sweep: high pierced above the prior high but closed back below it (failed
    # breakout) -- the classic "stop hunt above the highs" pattern, and its mirror below.
    feats["btc_liquidity_sweep_high"] = (
        (high > prior_high) & (close < prior_high)
    ).astype(int)
    feats["btc_liquidity_sweep_low"] = (
        (low < prior_low) & (close > prior_low)
    ).astype(int)

    # Equal highs: how many of the past `window` days' highs sit within `eq_tolerance`
    # (relative) of that window's maximum -- a rough proxy for liquidity resting at a
    # shared price level.
    def _count_near_max(values):
        m = values.max()
        if m == 0:
            return 0
        return np.sum(np.abs(values - m) / m <= eq_tolerance)

    feats["btc_equal_highs_count"] = high.rolling(window).apply(_count_near_max, raw=True)

    return feats


smc_features = add_smc_liquidity_features(merged, "btc")
print(smc_features.shape)
smc_features.tail()


In [ ]:
def add_cross_asset_features(df, asset_names):
    feats = pd.DataFrame(index=df.index)
    for name in asset_names:
        close_col = f"{name}_close"
        if close_col not in df.columns:
            continue
        daily_return = df[close_col].pct_change()
        # Lag by 1 day: today's feature = yesterday's return (conservative, leakage-safe)
        feats[f"{name}_return_lag1"] = daily_return.shift(1)
    return feats


cross_asset_names = [n for n in TICKERS.values() if n != "btc"]
cross_features = add_cross_asset_features(merged, cross_asset_names)
print(cross_features.shape)
cross_features.tail()


### Crypto-specific sentiment features (Funding Rate & Fear/Greed)

Same lagging logic as the cross-asset features above: today's feature = yesterday's value, so
nothing from "today" (which is used to build tomorrow's target) leaks in.

In [ ]:
def add_sentiment_features(df):
    feats = pd.DataFrame(index=df.index)
    if "funding_rate_avg" in df.columns:
        feats["funding_rate_lag1"] = df["funding_rate_avg"].shift(1)
    if "fear_greed_value" in df.columns:
        feats["fear_greed_lag1"] = df["fear_greed_value"].shift(1)
        # Day-over-day change in sentiment can matter as much as the level itself
        feats["fear_greed_change_lag1"] = df["fear_greed_value"].diff().shift(1)
    return feats


sentiment_features = add_sentiment_features(merged)
print(sentiment_features.shape)
sentiment_features.tail()


## 4. Target construction (leakage-safe)

The target for the row at date *t* is defined using the candle of date **t+1**:

```
target(t) = 1  if  Close(t+1) - Open(t+1) > 0
target(t) = 0  otherwise
```

All the features computed above for row *t* only use data available **up to and including**
day *t*, so there is no leakage from the future into the inputs.

In [ ]:
target = (merged["btc_close"].shift(-1) - merged["btc_open"].shift(-1) > 0).astype(int)
target.name = "target"

dataset = pd.concat(
    [btc_features, smc_features, cross_features, sentiment_features, target], axis=1
)

# The last row has no "tomorrow" to compute a target for -> drop it.
# Early rows have NaN from rolling windows / lagging -> drop them too.
dataset = dataset.dropna()

print("Final dataset shape:", dataset.shape)
dataset.head()


## 5. Train / test split (time-aware)

Financial time series must **never** be split randomly — that would let the model "see"
future patterns during training. We use a simple chronological split (80% train / 20% test),
and later also validate with `TimeSeriesSplit` for a more robust estimate.

In [ ]:
feature_cols = [c for c in dataset.columns if c != "target"]
X = dataset[feature_cols]
y = dataset["target"]

split_idx = int(len(dataset) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape[0]} rows ({X_train.index.min().date()} -> {X_train.index.max().date()})")
print(f"Test:  {X_test.shape[0]} rows ({X_test.index.min().date()} -> {X_test.index.max().date()})")


## 6. Baseline

Before trusting any model's accuracy, we need to know what a "dumb" model would score —
simply always predicting the majority class.

In [ ]:
baseline_rate = y_train.mean()
majority_class = int(baseline_rate > 0.5)
baseline_accuracy = max(baseline_rate, 1 - baseline_rate)

print(f"Share of green days in training set: {baseline_rate:.3f}")
print(f"Baseline accuracy (always predict class {majority_class}): {baseline_accuracy:.3f}")
print("\nAny model below this line is worse than doing nothing.")


## 7. Model training

We train all three boosting libraries with the same features and a chronological
`TimeSeriesSplit` for cross-validation during training, then evaluate all of them on the
same held-out test set.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

models = {
    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
    "CatBoost": CatBoostClassifier(
        n_estimators=200,
        learning_rate=0.05,
        depth=4,
        verbose=0,
        random_state=RANDOM_STATE,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbose=-1,
    ),
}

fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    print(f"{name} trained.")


## 8. Evaluation & comparison

In [ ]:
results = []
predictions = {}

for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    predictions[name] = (y_pred, y_proba)

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).set_index("model")
results_df.loc["Baseline (majority class)"] = [baseline_accuracy, np.nan, np.nan, np.nan, np.nan]
results_df.round(3)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, fitted_models.keys()):
    y_pred, _ = predictions[name]
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Red", "Green"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()


## 9. Feature importance

Understanding *which* features drive each model's decisions is important both for
interpretability and for spotting anything suspicious (e.g. a feature with unrealistically
high importance can be a sign of leakage).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, model) in zip(axes, fitted_models.items()):
    importances = pd.Series(model.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=True).tail(15)
    importances.plot(kind="barh", ax=ax)
    ax.set_title(f"{name} - top 15 features")

plt.tight_layout()
plt.show()


## 10. Simple backtest simulation

⚠️ **Educational only — not investment advice.** We simulate a naive strategy: go long for
one day whenever the model predicts "green", stay in cash otherwise. We compare the
cumulative return of this strategy against simple Buy & Hold over the test period, and report
the Sharpe ratio. This ignores trading fees, slippage, and taxes, which would meaningfully
reduce real-world returns.

In [ ]:
def backtest(y_pred, actual_returns, label):
    strategy_returns = actual_returns * y_pred  # only "earn" the day's return when predicted green
    cumulative = (1 + strategy_returns).cumprod()
    sharpe = (
        strategy_returns.mean() / strategy_returns.std() * np.sqrt(365)
        if strategy_returns.std() > 0 else np.nan
    )
    return cumulative, sharpe


# Actual next-day return over the test period, aligned with our target's definition
actual_next_day_return = merged["btc_close"].pct_change().shift(-1).reindex(X_test.index)

plt.figure(figsize=(12, 6))
buy_hold_cum = (1 + actual_next_day_return).cumprod()
plt.plot(buy_hold_cum, label="Buy & Hold", linewidth=2, linestyle="--")

for name in fitted_models.keys():
    y_pred, _ = predictions[name]
    cum, sharpe = backtest(y_pred, actual_next_day_return, name)
    plt.plot(cum, label=f"{name} (Sharpe={sharpe:.2f})")

plt.title("Cumulative return: model-driven strategy vs Buy & Hold (test period)")
plt.ylabel("Cumulative growth of $1")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Bonus: predicting next-day volatility regime

Predicting **direction** is close to a coin flip because daily crypto returns behave close to
a random walk. **Volatility**, on the other hand, is well known in financial research to
exhibit **clustering** — a well-documented "stylized fact" (large moves tend to be followed by
large moves, calm periods by calm periods; this is the intuition behind GARCH-family models).
So this is a genuinely more tractable target to try the same three models on.

### Target definition (causal, leakage-safe)
```
threshold(t)  = trailing 90-day median of |daily return|, using data up to and including day t
target_vol(t) = 1  if  |return(t+1)| > threshold(t)
target_vol(t) = 0  otherwise
```
The threshold only ever uses **past** data (a trailing rolling median), so it's known at the
end of day *t* — exactly like every other feature in this notebook. We reuse the exact same
feature set (`feature_cols`) as before; only the target changes.

In [ ]:
abs_return = merged["btc_close"].pct_change().abs()
vol_threshold = abs_return.rolling(90).median()

target_vol = (abs_return.shift(-1) > vol_threshold).astype(int)
target_vol.name = "target_vol"

vol_dataset = pd.concat(
    [btc_features, smc_features, cross_features, sentiment_features, target_vol], axis=1
).dropna()

X_vol = vol_dataset[feature_cols]
y_vol = vol_dataset["target_vol"]

split_idx_vol = int(len(vol_dataset) * 0.8)
X_train_vol, X_test_vol = X_vol.iloc[:split_idx_vol], X_vol.iloc[split_idx_vol:]
y_train_vol, y_test_vol = y_vol.iloc[:split_idx_vol], y_vol.iloc[split_idx_vol:]

vol_baseline_rate = y_train_vol.mean()
vol_baseline_accuracy = max(vol_baseline_rate, 1 - vol_baseline_rate)
print(f"Train: {X_train_vol.shape[0]} rows, Test: {X_test_vol.shape[0]} rows")
print(f"Baseline accuracy (volatility task): {vol_baseline_accuracy:.3f}")


In [ ]:
vol_models = {
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
    "CatBoost": CatBoostClassifier(
        n_estimators=200, learning_rate=0.05, depth=4,
        verbose=0, random_state=RANDOM_STATE,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbose=-1,
    ),
}

vol_results = []
for name, model in vol_models.items():
    model.fit(X_train_vol, y_train_vol)
    y_pred_vol = model.predict(X_test_vol)
    y_proba_vol = model.predict_proba(X_test_vol)[:, 1]

    vol_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test_vol, y_pred_vol),
        "precision": precision_score(y_test_vol, y_pred_vol),
        "recall": recall_score(y_test_vol, y_pred_vol),
        "f1": f1_score(y_test_vol, y_pred_vol),
        "roc_auc": roc_auc_score(y_test_vol, y_proba_vol),
    })

vol_results_df = pd.DataFrame(vol_results).set_index("model")
vol_results_df.loc["Baseline (majority class)"] = [vol_baseline_accuracy, np.nan, np.nan, np.nan, np.nan]
vol_results_df.round(3)


Compare `vol_results_df` above against the direction-prediction `results_df` from section 8.
If the volatility-regime models show a meaningfully higher ROC-AUC (further from 0.5) than the
direction models did, that supports the idea that volatility clustering carries more usable
signal than daily price direction — a much more defensible, publishable finding than a
marginal edge on direction.

## 12. Conclusions & limitations

- Report both comparison tables (`results_df` for direction, `vol_results_df` for volatility
  regime) and note whether either model meaningfully beat its baseline.
- **Limitations:**
  - No on-chain data (exchange netflows, whale activity) or order-book microstructure.
  - Daily granularity hides a lot of information available at higher (hourly/minute)
    frequency.
  - Markets evolve — a model trained on 2018-2024 data is not guaranteed to keep working in
    future regimes (this is called *concept drift*), so any real deployment would need
    periodic retraining and monitoring.
  - Backtest ignores transaction costs, slippage, and taxes.
  - The Smart Money Concepts liquidity features and the funding-rate/fear-greed features are
    based on informal or crypto-market-specific heuristics rather than universally agreed
    financial theory; treat them as exploratory.
- **Possible future extensions:** walk-forward re-training, hyperparameter tuning via
  `TimeSeriesSplit` grid/Bayesian search, ensembling the three models, confidence-based
  filtering (only act on the most confident predictions), on-chain metrics, extending to
  intraday data.
